# Chapter 4 — How Do You Measure a Hallucination?

**Book alignment:** Hallucination From First Principles, Chapter 4

**Question this notebook isolates:** Do proximity sensors stay high where a directional check flips on negation and relation-reversal pairs?

Synthetic fixtures in this notebook demonstrate the mechanism type only and do not reproduce the book's 10k-row empirical run.


In [ ]:
import numpy as np

rng = np.random.default_rng(3)


## 1. Proximity stays high where entailment flips

Chapter 4 (sec. 8-9) separates symmetric proximity (lexical overlap, embedding cosine) from the directional claim-evidence relation (NLI). We score synthetic claim/evidence pairs with count-vector cosine plus jaccard overlap against a rule-based asymmetric entailment stub: proximity should stay high on negation and reversal pairs while the stub flips to CONTRADICTION.


In [ ]:
def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb)


def bow_cosine(a, b):
    vocab = sorted(set(a.lower().split()) | set(b.lower().split()))
    va = np.array([a.lower().split().count(w) for w in vocab], dtype=float)
    vb = np.array([b.lower().split().count(w) for w in vocab], dtype=float)
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


def nli_stub(pair):
    """Asymmetric stub: premise=evidence, hypothesis=claim."""
    if pair["polarity_ev"] != pair["polarity_cl"]:
        return "CONTRADICTION"
    if pair["roles_ev"] != pair["roles_cl"]:
        return "CONTRADICTION"
    if pair["strength"] == "inflated":
        return "NEUTRAL"  # associated -> caused: topical but unlicensed
    return "ENTAILMENT"


pairs = [
    {"name": "negation_flip", "ev": "the study did not find a statistically significant reduction in mortality in the trial population during the six month follow up", "cl": "the study found a statistically significant reduction in mortality in the trial population during the six month follow up", "polarity_ev": "neg", "polarity_cl": "pos", "roles_ev": ("study", "find", "reduction"), "roles_cl": ("study", "find", "reduction"), "strength": "same"},
    {"name": "role_reversal", "ev": "Company A acquired Company B from Company C in 2024", "cl": "Company B acquired Company A from Company C in 2024", "polarity_ev": "pos", "polarity_cl": "pos", "roles_ev": ("A", "acquired", "B"), "roles_cl": ("B", "acquired", "A"), "strength": "same"},
    {"name": "paraphrase_ok", "ev": "Company A acquired Company B from Company C in 2024", "cl": "in 2024 Company A bought Company B from Company C", "polarity_ev": "pos", "polarity_cl": "pos", "roles_ev": ("A", "acquired", "B"), "roles_cl": ("A", "acquired", "B"), "strength": "same"},
    {"name": "causal_inflation", "ev": "the treatment was associated with lower symptoms in one small study", "cl": "the treatment is an established cure for the disease", "polarity_ev": "pos", "polarity_cl": "pos", "roles_ev": ("treatment", "associated", "symptoms"), "roles_cl": ("treatment", "associated", "symptoms"), "strength": "inflated"},
]

print(f"{'pair':>16} | {'jaccard':>7} {'cosine':>6} {'nli':>13}")
rows = {}
for p in pairs:
    j, c, n = jaccard(p["ev"], p["cl"]), bow_cosine(p["ev"], p["cl"]), nli_stub(p)
    rows[p["name"]] = (j, c, n)
    print(f"{p['name']:>16} | {j:7.3f} {c:6.3f} {n:>13}")


In [ ]:
for name in ("negation_flip", "role_reversal"):
    j, c, n = rows[name]
    assert j > 0.70, (name, j)  # proximity stays high
    assert c > 0.90, (name, c)
    assert n == "CONTRADICTION", (name, n)  # ... while direction flips
assert rows["paraphrase_ok"][2] == "ENTAILMENT"
assert rows["causal_inflation"][2] == "NEUTRAL"  # proximity without licensing
print("proxy gap confirmed: high proximity coexists with CONTRADICTION")


## 2. Check the trace first, guess last

Chapter 4 (sec. 6) orders sensors: when authoritative system state exists, compare against the trace before reaching for probabilistic semantics. We stub a runtime trace plus a fluent-but-wrong semantic score and show the trace verdict overriding the semantic guess on tool/file claims.


In [ ]:
trace = {"tool_calls": [], "attachments": [], "tests": {"ran": False, "passed": 0, "total": 0}}
trace_ok = {"tool_calls": ["run_tests"], "attachments": ["report.pdf"], "tests": {"ran": True, "passed": 214, "total": 214}}


def trace_check(kind, detail, tr):
    if kind == "tool_claim":
        return "PASS" if detail in tr["tool_calls"] else "FAIL"
    if kind == "attachment_claim":
        return "PASS" if detail in tr["attachments"] else "FAIL"
    if kind == "test_claim":
        t = tr["tests"]
        return "PASS" if (t["ran"] and (t["passed"], t["total"]) == detail) else "FAIL"
    raise ValueError(kind)


def semantic_plausibility(_claim):
    return 0.88  # stub: fluent false runtime claims still look plausible


claims = [
    {"name": "phantom_tool", "kind": "tool_claim", "detail": "run_tests", "text": "I ran the test suite, all 214 tests passed"},
    {"name": "phantom_pdf", "kind": "attachment_claim", "detail": "report.pdf", "text": "I inspected the attached PDF"},
]
print(f"{'claim':>12} | {'trace':>5} {'semantic':>8} {'route':>7}")
verdicts = {}
for clm in claims:
    t = trace_check(clm["kind"], clm["detail"], trace)
    s = semantic_plausibility(clm["text"])
    route = "REJECT" if t == "FAIL" else "CONSIDER"
    verdicts[clm["name"]] = (t, s, route)
    print(f"{clm['name']:>12} | {t:>5} {s:8.2f} {route:>7}")

control = trace_check("test_claim", (214, 214), trace_ok)
print("control with matching trace:", control)
record = {"support": {"sensor": "nli_stub_v1", "value": rows["role_reversal"][2]}, "runtime_state": {"sensor": "trace_assertion_v1", "value": verdicts["phantom_tool"][0]}}
print("typed record:", record)


In [ ]:
for name in ("phantom_tool", "phantom_pdf"):
    t, s, route = verdicts[name]
    assert t == "FAIL"  # authoritative state: event never happened
    assert s > 0.80  # fluency/plausibility stays high anyway
    assert route == "REJECT"  # trace overrides the semantic guess
assert control == "PASS"
assert set(record) == {"support", "runtime_state"}  # typed entries, not one scalar
print("trace-first routing confirmed: FAIL trace rejects fluent false claims")


## What we earned

- Negation and role-reversal pairs keep jaccard above 0.70 and count-vector cosine above 0.90 while the directional stub flips to CONTRADICTION: proximity is non-directional, support is the target relation.
- Fluent false runtime claims score 0.88 on the semantic stub yet FAIL the trace check, so trace-first routing rejects them; the typed record preserves both signals instead of averaging them.

Next: Chapter 5 — Hallucination Energy, which turns evidence-set containment into a narrow geometric sensor and tests where it breaks.
